In [ ]:
!pip install rdkit networkx ortools pubchempy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.8/31.8 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [ ]:
# ==========================================================
# IMPORTS
# ==========================================================

from rdkit import Chem
import networkx as nx
from ortools.sat.python import cp_model
import pubchempy as pcp
import time

In [ ]:
# ==========================================================
# SMILES -> MOLECULAR GRAPH
# ==========================================================

def smiles_to_graph(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:
        raise ValueError("Invalid SMILES")

    mol = Chem.RemoveHs(mol)

    G = nx.Graph()

    # Atoms = vertices
    for atom in mol.GetAtoms():
        G.add_node(atom.GetIdx())

    # Bonds = edges
    for bond in mol.GetBonds():

        u = bond.GetBeginAtomIdx()
        v = bond.GetEndAtomIdx()

        G.add_edge(u, v)

    return G



In [ ]:
# ==========================================================
# PUBCHEM COMPOUND NAME
# ==========================================================

def get_compound_name(smiles):

    try:

        compounds = pcp.get_compounds(
            smiles,
            namespace="smiles"
        )

        if not compounds:
            return "Name not found in PubChem"

        compound = compounds[0]

        if getattr(compound, "iupac_name", None):

            return compound.iupac_name

        synonyms = getattr(
            compound,
            "synonyms",
            None
        )

        if synonyms:

            return synonyms[0]

        return "Name not found in PubChem"

    except Exception as e:

        return f"PubChem lookup failed: {e}"



In [ ]:
# ==========================================================
# 1. DOMINATION NUMBER
# ==========================================================

def domination_number_cpsat(G, time_limit=60):

    nodes = list(G.nodes())
    n = len(nodes)

    model = cp_model.CpModel()

    x = {
        v: model.NewBoolVar(
            f"dom_x_{v}"
        )
        for v in nodes
    }

    # ------------------------------------------------------
    # Domination
    # ------------------------------------------------------

    for v in nodes:

        closed_neighborhood = [
            v
        ] + list(G.neighbors(v))

        model.Add(
            sum(
                x[u]
                for u in closed_neighborhood
            ) >= 1
        )

    # ------------------------------------------------------
    # Objective
    # ------------------------------------------------------

    total_selected = model.NewIntVar(
        1,
        n,
        "dom_total_selected"
    )

    model.Add(
        total_selected ==
        sum(x[v] for v in nodes)
    )

    model.Minimize(total_selected)

    # ------------------------------------------------------
    # Solver
    # ------------------------------------------------------

    solver = cp_model.CpSolver()

    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.num_search_workers = 8

    start = time.time()

    status = solver.Solve(model)

    elapsed = time.time() - start

    if status == cp_model.OPTIMAL:

        return (
            solver.Value(total_selected),
            f"OPTIMAL | {elapsed:.4f} seconds"
        )

    elif status == cp_model.FEASIBLE:

        return (
            solver.Value(total_selected),
            f"FEASIBLE ONLY | {elapsed:.4f} seconds"
        )

    elif status == cp_model.UNKNOWN:

        return (
            None,
            f"NOT VERIFIED | {elapsed:.4f} seconds"
        )

    return None, "No solution"


In [ ]:
# ==========================================================
# 2. TOTAL DOMINATION NUMBER
# ==========================================================

def total_domination_number_cpsat(G, time_limit=60):

    nodes = list(G.nodes())
    n = len(nodes)

    model = cp_model.CpModel()

    x = {
        v: model.NewBoolVar(
            f"td_x_{v}"
        )
        for v in nodes
    }

    # ------------------------------------------------------
    # Every vertex needs a selected neighbor
    # ------------------------------------------------------

    for v in nodes:

        neighbors = list(G.neighbors(v))

        if not neighbors:

            return None, "No total dominating set"

        model.Add(
            sum(
                x[u]
                for u in neighbors
            ) >= 1
        )

    # ------------------------------------------------------
    # Objective
    # ------------------------------------------------------

    total_selected = model.NewIntVar(
        1,
        n,
        "td_total_selected"
    )

    model.Add(
        total_selected ==
        sum(x[v] for v in nodes)
    )

    model.Minimize(total_selected)

    # ------------------------------------------------------
    # Solver
    # ------------------------------------------------------

    solver = cp_model.CpSolver()

    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.num_search_workers = 8

    start = time.time()

    status = solver.Solve(model)

    elapsed = time.time() - start

    if status == cp_model.OPTIMAL:

        return (
            solver.Value(total_selected),
            f"OPTIMAL | {elapsed:.4f} seconds"
        )

    elif status == cp_model.FEASIBLE:

        return (
            solver.Value(total_selected),
            f"FEASIBLE ONLY | {elapsed:.4f} seconds"
        )

    elif status == cp_model.UNKNOWN:

        return (
            None,
            f"NOT VERIFIED | {elapsed:.4f} seconds"
        )

    return None, "No solution"


In [ ]:
# ==========================================================
# 3. POWER DOMINATION NUMBER
# ==========================================================
#
# This follows the same definition as your working
# brute-force implementation.
#
# Initial:
#       observed = N[S]
#
# Propagation:
#       if an observed vertex has exactly ONE
#       unobserved neighbor, that neighbor is forced.
#
# ==========================================================

def power_domination_number_cpsat(
    G,
    time_limit=60
):

    nodes = list(G.nodes())
    n = len(nodes)

    model = cp_model.CpModel()

    # ------------------------------------------------------
    # Selected vertices
    # ------------------------------------------------------

    x = {
        v: model.NewBoolVar(
            f"pd_x_{v}"
        )
        for v in nodes
    }

    total_selected = model.NewIntVar(
        1,
        n,
        "pd_total_selected"
    )

    model.Add(
        total_selected ==
        sum(x[v] for v in nodes)
    )

    # ------------------------------------------------------
    # observed[v,t]
    # ------------------------------------------------------

    observed = {}

    for t in range(n + 1):

        for v in nodes:

            observed[(v, t)] = model.NewBoolVar(
                f"pd_observed_{v}_{t}"
            )

    # ======================================================
    # INITIAL OBSERVATION
    # ======================================================

    for v in nodes:

        closed_neighborhood = [
            v
        ] + list(G.neighbors(v))

        # Let z represent whether N[v] contains
        # at least one selected vertex.

        initial_sum = sum(
            x[u]
            for u in closed_neighborhood
        )

        # If any selected vertex is in N[v],
        # v must be observed.
        model.Add(
            initial_sum >= 1
        ).OnlyEnforceIf(
            observed[(v, 0)]
        )

        # If v is NOT observed, no selected vertex
        # can be in N[v].
        model.Add(
            initial_sum == 0
        ).OnlyEnforceIf(
            observed[(v, 0)].Not()
        )

    # ======================================================
    # FORCE VARIABLES
    # ======================================================

    force = {}

    for t in range(n):

        for v in nodes:

            neighbors = list(G.neighbors(v))

            for u in neighbors:

                force[(v, u, t)] = model.NewBoolVar(
                    f"pd_force_{v}_{u}_{t}"
                )

    # ======================================================
    # EXACT FORCE CONDITION
    # ======================================================

    for t in range(n):

        for v in nodes:

            neighbors = list(G.neighbors(v))

            for u in neighbors:

                f = force[(v, u, t)]

                # --------------------------------------------------
                # f -> v observed
                # --------------------------------------------------

                model.AddImplication(
                    f,
                    observed[(v, t)]
                )

                # --------------------------------------------------
                # f -> u unobserved
                # --------------------------------------------------

                model.AddImplication(
                    f,
                    observed[(u, t)].Not()
                )

                # --------------------------------------------------
                # f -> every other neighbor observed
                # --------------------------------------------------

                for w in neighbors:

                    if w != u:

                        model.AddImplication(
                            f,
                            observed[(w, t)]
                        )

                # --------------------------------------------------
                # Reverse direction
                #
                # If:
                #   v observed
                #   u unobserved
                #   all other neighbors observed
                #
                # then f must be TRUE.
                # --------------------------------------------------

                clause = [
                    observed[(v, t)].Not(),
                    observed[(u, t)]
                ]

                for w in neighbors:

                    if w != u:

                        clause.append(
                            observed[(w, t)].Not()
                        )

                clause.append(f)

                model.AddBoolOr(clause)

    # ======================================================
    # OBSERVATION UPDATE
    # ======================================================

    for t in range(n):

        for u in nodes:

            incoming_forces = [
                force[(v, u, t)]
                for v in G.neighbors(u)
            ]

            # --------------------------------------------------
            # Already observed -> remains observed
            # --------------------------------------------------

            model.AddImplication(
                observed[(u, t)],
                observed[(u, t + 1)]
            )

            # --------------------------------------------------
            # A force -> observed
            # --------------------------------------------------

            for f in incoming_forces:

                model.AddImplication(
                    f,
                    observed[(u, t + 1)]
                )

            # --------------------------------------------------
            # CRITICAL:
            #
            # observed[t+1] can ONLY happen because:
            #
            #   already observed
            #
            # OR
            #
            #   forced by a neighbor.
            # --------------------------------------------------

            possible_sources = [
                observed[(u, t)]
            ] + incoming_forces

            model.Add(
                sum(possible_sources)
                >=
                observed[(u, t + 1)]
            )

    # ======================================================
    # ALL VERTICES MUST EVENTUALLY BE OBSERVED
    # ======================================================

    for v in nodes:

        model.Add(
            observed[(v, n)] == 1
        )

    # ======================================================
    # OBJECTIVE
    # ======================================================

    model.Minimize(total_selected)

    # ======================================================
    # SOLVER
    # ======================================================

    solver = cp_model.CpSolver()

    solver.parameters.max_time_in_seconds = time_limit
    solver.parameters.num_search_workers = 8

    start = time.time()

    status = solver.Solve(model)

    elapsed = time.time() - start

    # ======================================================
    # RESULT
    # ======================================================

    if status == cp_model.OPTIMAL:

        return (
            solver.Value(total_selected),
            f"OPTIMAL | {elapsed:.4f} seconds"
        )

    elif status == cp_model.FEASIBLE:

        return (
            solver.Value(total_selected),
            f"FEASIBLE ONLY | {elapsed:.4f} seconds"
        )

    elif status == cp_model.UNKNOWN:

        return (
            None,
            f"NOT VERIFIED | {elapsed:.4f} seconds"
        )

    return None, "No solution"



In [ ]:
# ==========================================================
# MAIN PROGRAM
# ==========================================================

smiles = input("Enter SMILES: ").strip()


# ==========================================================
# VALIDATE SMILES
# ==========================================================

try:

    G = smiles_to_graph(smiles)

except ValueError as e:

    print("\nError:", e)
    raise SystemExit


# ==========================================================
# PUBCHEM
# ==========================================================

print("\nLooking up compound in PubChem...")

compound_name = get_compound_name(smiles)


# ==========================================================
# MOLECULAR GRAPH INFORMATION
# ==========================================================

print("\n==========================================")
print("          MOLECULAR GRAPH")
print("==========================================")

print(
    "Compound name:",
    compound_name
)

print(
    "Number of vertices:",
    G.number_of_nodes()
)

print(
    "Number of edges:",
    G.number_of_edges()
)


# ==========================================================
# CALCULATE SIX PARAMETERS
# ==========================================================

print("\n==========================================")
print("       CALCULATING PARAMETERS")
print("==========================================")

overall_start = time.time()


# ----------------------------------------------------------
# 1. Domination
# ----------------------------------------------------------

print("\n[1/3] Calculating domination number...")

gamma, status_gamma = domination_number_cpsat(
    G,
    time_limit=60
)


# ----------------------------------------------------------
# 2. Total domination
# ----------------------------------------------------------

print("\n[2/3] Calculating total domination number...")

gamma_t, status_gamma_t = total_domination_number_cpsat(
    G,
    time_limit=60
)


# ----------------------------------------------------------
# 3. Power domination
# ----------------------------------------------------------

print("\n[3/3] Calculating power domination number...")

gamma_p, status_gamma_p = power_domination_number_cpsat(
    G,
    time_limit=60
)

# ==========================================================
# TOTAL TIME
# ==========================================================

overall_elapsed = time.time() - overall_start


# ==========================================================
# FINAL RESULTS
# ==========================================================

print("\n\n==========================================")
print("             FINAL RESULTS")
print("==========================================")

print(
    "Compound name:",
    compound_name
)

print(
    "Number of vertices:",
    G.number_of_nodes()
)

print(
    "Number of edges:",
    G.number_of_edges()
)

print("------------------------------------------")

print(
    "Domination number γ(G):",
    gamma
)

print(
    "Status:",
    status_gamma
)

print("------------------------------------------")

print(
    "Total domination number γₜ(G):",
    gamma_t
)

print(
    "Status:",
    status_gamma_t
)

print("------------------------------------------")

print(
    "Power domination number γₚ(G):",
    gamma_p
)

print(
    "Status:",
    status_gamma_p
)

print("------------------------------------------")

print(
    "Total calculation time:",
    round(
        overall_elapsed,
        4
    ),
    "seconds"
)

print("==========================================")

Enter SMILES: CS(=O)(=O)OCCCCOS(=O)(=O)C

Looking up compound in PubChem...

          MOLECULAR GRAPH
Compound name: 4-methylsulfonyloxybutyl methanesulfonate
Number of vertices: 14
Number of edges: 13

       CALCULATING PARAMETERS

[1/3] Calculating domination number...

[2/3] Calculating total domination number...

[3/3] Calculating power domination number...


             FINAL RESULTS
Compound name: 4-methylsulfonyloxybutyl methanesulfonate
Number of vertices: 14
Number of edges: 13
------------------------------------------
Domination number γ(G): 4
Status: OPTIMAL | 0.0130 seconds
------------------------------------------
Total domination number γₜ(G): 6
Status: OPTIMAL | 0.0067 seconds
------------------------------------------
Power domination number γₚ(G): 2
Status: OPTIMAL | 0.1036 seconds
------------------------------------------
Total calculation time: 0.1617 seconds
